In [1]:
import importlib
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # notebooks -> DEM_MCM
sys.path.insert(0, str(project_root))

import dem_mcm_coupling.bucket_io as b_io
import sympy as sp
from dem_mcm_coupling import analyze_results
from dem_mcm_coupling import partitioners as part
from dem_mcm_coupling import run_sweep as r_s

sp.init_printing()
importlib.reload(analyze_results)
importlib.reload(r_s)
importlib.reload(b_io)
importlib.reload(part)

<module 'src.partitioners' from '/teamspace/studios/this_studio/MyStudio/DEM_MCM/src/partitioners.py'>

In [2]:
analyzer = analyze_results.MarkovAnalyzer()

In [3]:
analyzer.load_method("cartesian")

   ✅ cartesian_nx2_ny2_nz2_NLT10_step10_dt2_tau50_start250: shape=(8, 8)
   ✅ cartesian_nx2_ny2_nz2_NLT10_step20_dt2_tau50_start250: shape=(8, 8)
   ✅ cartesian_nx3_ny2_nz3_NLT10_step100_dt2_tau50_start250: shape=(18, 18)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step100_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step10_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step20_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step30_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step40_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx3_ny3_nz3_NLT10_step50_dt2_tau50_start250: shape=(27, 27)
   ✅ cartesian_nx4_ny2_nz4_NLT10_step100_dt2_tau50_start250: shape=(32, 32)
   ✅ cartesian_nx4_ny3_nz4_NLT10_step10_dt2_tau50_start250: shape=(48, 48)
   ✅ cartesian_nx4_ny3_nz4_NLT10_step20_dt2_tau50_start250: shape=(48, 48)
   ✅ cartesian_nx4_ny3_nz4_NLT10_step50_dt2_tau50_start250: shape=(48, 48)
   ✅ cartesian_nx4_ny4_nz4

In [4]:
results = analyzer.results

In [5]:
experiments = [
    i
    for i in analyzer._list_folders()
    if i.startswith("cartesian_") and "_step10_" in i
]
experiments

['cartesian_nx2_ny2_nz2_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx3_ny3_nz3_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx4_ny3_nz4_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx4_ny4_nz4_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx5_ny3_nz5_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx5_ny5_nz5_NLT10_step10_dt2_tau100_start250',
 'cartesian_nx5_ny5_nz5_NLT10_step10_dt2_tau200_start250',
 'cartesian_nx5_ny5_nz5_NLT10_step10_dt2_tau20_start250',
 'cartesian_nx5_ny5_nz5_NLT10_step10_dt2_tau50_start250',
 'cartesian_nx5_ny5_nz5_NLT1_step10_dt2_tau50_start250',
 'cartesian_nx5_ny5_nz5_NLT2_step10_dt2_tau50_start250',
 'cartesian_nx5_ny5_nz5_NLT3_step10_dt2_tau50_start250',
 'cartesian_nx5_ny5_nz5_NLT5_step10_dt2_tau50_start250']

In [14]:
M = [results[experiments[i]]["matrix"] for i in range(len(experiments))]
# M[-1].sum(axis=0)
M[-1].all()

False

Nous constatons que pour un découpage cartésien, la condition de normalisation n'est pas respecté pour un nombre de partitions au delà de 8 et pour un step de 10 et un dt de 2. On observe cela par des nan dans la somme de lignes des colonnes de chaque colonne.

Pour resoudre ce problème nous devons faire plus d'observations et ce en augmentant le NLT, le step et en reduisant le dt

In [49]:
configs = [
    r_s.ExperimentConfig(
        "catesian", {"nx": 5, "ny": 5, "nz": 5}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 4, "ny": 4, "nz": 4}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 3, "ny": 3, "nz": 3}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 2, "ny": 2, "nz": 2}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 6, "ny": 6, "nz": 6}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 7, "ny": 7, "nz": 7}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 8, "ny": 8, "nz": 8}, nlt=10, dt=2, step=20
    ),
    r_s.ExperimentConfig(
        "catesian", {"nx": 9, "ny": 9, "nz": 9}, nlt=10, dt=2, step=20
    ),
]

In [50]:
r_s.run_markov_sweep("cartesian", configs=configs)

  SWEEP MARKOVIEN — méthode: CARTESIAN
🖥️  Device: cpu


HTTP Error 504 thrown while requesting GET https://huggingface.co/api/buckets/ktongue/DEM_MCM/tree/Output%20Paraview?limit=1000&recursive=true&cursor=eyJwYXRoIjoiT3V0cHV0IFBhcmF2aWV3Ly5jYWNoZS9odWdnaW5nZmFjZS91cGxvYWQvZGF0YV8xNDgwLmNzdi5sb2NrIn0%3D
Retrying in 1s [Retry 1/5].


KeyboardInterrupt: 

In [7]:
experiments = [
    i
    for i in analyzer._list_folders()
    if i.startswith("cartesian_") and "_step20_" in i and "dt1_" in i and "tau50_" in i
]
experiments

['cartesian_nx5_ny5_nz5_NLT10_step20_dt1_tau50_start250']

In [15]:
M = [results[experiments[i]]["matrix"] for i in range(len(experiments))]
# M[0].sum(axis=0)
M[0].all()

False

Nous constatons que malgrés le fait que nous étendons le temps d'observation global(NLT,step,dt), la matrice de transition construite ne respecte pas toujours les conditions de normalisation

Le découpage en cartésien crée des partitions pour lesquelles les particules ne transitent pas. Pour capturer leur transitions, il faudrait les observer sur une période très grande voire infinie

Une autre stratégie serait de découper en peu de partitions selon la direction y du mélangeur pour en effet eviter de tomber dans la partie vide du mélangeur et d'observer les particules sur une plus grande durée.

